# 02 — Merging FEMA Declarations with NOAA Storm Events

## Join logic
Each FEMA disaster covers multiple counties over a date range (incidentBeginDate → incidentEndDate).
We join NOAA events where:
1. The NOAA event's county FIPS appears in the set of counties affected by the FEMA disaster
2. The NOAA event's date range overlaps with the FEMA incident date range
   (i.e., NOAA begin_date <= FEMA end_date AND NOAA end_date >= FEMA begin_date)

## Aggregation
Since one FEMA disaster can match multiple NOAA events (many counties, multiple storms),
we aggregate NOAA features per disaster:
- Peak magnitude (max MAG across matched events) — proxy for storm intensity
- Total NOAA damage estimate (sum) — independent damage signal
- Max deaths/injuries — severity proxy
- Event count — breadth of physical impact

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# Phase 1 processed data (already cleaned)
FEMA_PROCESSED = '../../data/processed/'
PROCESSED      = '../data/processed/'
os.makedirs(PROCESSED, exist_ok=True)

## Load FEMA and NOAA Data
Load the Phase 1 cleaned disaster-level dataset and the filtered NOAA events from notebook 01.

In [ ]:
fema = pd.read_csv(FEMA_PROCESSED + 'cleaned_disaster_level.csv', low_memory=False)
noaa = pd.read_csv(PROCESSED + 'noaa_filtered.csv', low_memory=False)

# Parse dates
for col in ['incidentBeginDate', 'incidentEndDate', 'declarationDate']:
    if col in fema.columns:
        fema[col] = pd.to_datetime(fema[col], errors='coerce')

noaa['begin_date'] = pd.to_datetime(noaa['begin_date'], errors='coerce')
noaa['end_date']   = pd.to_datetime(noaa['end_date'],   errors='coerce')

print(f'FEMA disasters: {len(fema):,}')
print(f'NOAA events:    {len(noaa):,}')
print(f'FEMA columns:   {list(fema.columns)}')

## Join on State FIPS + Date Overlap
For each FEMA disaster, we find all NOAA events in the same state whose date range
overlaps the FEMA incident period. We then aggregate those matched events into
summary statistics per disaster.

**Note:** The join is state-level (not county-level) because FEMA declaration records
are aggregated to the disaster level rather than individual county rows. A county-level
join would require the county-level FEMA PA project data, which is Phase 1 county pipeline.

In [ ]:
# FEMA state FIPS
fema['state_fips'] = fema['stateNumberCode'].astype(str).str.zfill(2)

noaa_agg_rows = []

for _, disaster in fema.iterrows():
    s_fips     = disaster['state_fips']
    inc_begin  = disaster['incidentBeginDate']
    inc_end    = disaster['incidentEndDate']
    
    if pd.isna(inc_begin):
        noaa_agg_rows.append({'disasterNumber': disaster['disasterNumber']})
        continue
    
    if pd.isna(inc_end):
        inc_end = inc_begin + pd.Timedelta(days=30)
    
    # Match NOAA events: same state, dates overlap
    mask = (
        (noaa['state_fips'] == s_fips) &
        (noaa['begin_date'] <= inc_end) &
        (noaa['end_date']   >= inc_begin)
    )
    matched = noaa[mask]
    
    if len(matched) == 0:
        noaa_agg_rows.append({'disasterNumber': disaster['disasterNumber']})
        continue
    
    # Filter MAG to wind/water events only (non-null numeric)
    mag_vals = pd.to_numeric(matched['mag'], errors='coerce').dropna()
    
    noaa_agg_rows.append({
        'disasterNumber':        disaster['disasterNumber'],
        'noaa_event_count':      len(matched),
        'noaa_peak_magnitude':   mag_vals.max() if len(mag_vals) > 0 else np.nan,
        'noaa_mean_magnitude':   mag_vals.mean() if len(mag_vals) > 0 else np.nan,
        'noaa_total_damage_usd': matched['total_damage_usd'].sum(),
        'noaa_max_deaths':       matched[['deaths_direct','deaths_indirect']].sum(axis=1).max(),
        'noaa_total_deaths':     matched[['deaths_direct','deaths_indirect']].sum(axis=1).sum(),
        'noaa_max_injuries':     matched[['injuries_direct','injuries_indirect']].sum(axis=1).max(),
        'noaa_matched':          1
    })

noaa_agg = pd.DataFrame(noaa_agg_rows)
noaa_agg['noaa_matched'] = noaa_agg['noaa_matched'].fillna(0).astype(int)

merged = fema.merge(noaa_agg, on='disasterNumber', how='left')

match_rate = noaa_agg['noaa_matched'].mean()
print(f'Disasters with NOAA match: {noaa_agg["noaa_matched"].sum():,} / {len(fema):,} ({match_rate:.1%})')
print(f'Merged shape: {merged.shape}')

## Save Merged Dataset
Save the merged FEMA + NOAA dataset for cleaning and feature engineering in notebook 03.

In [ ]:
merged.to_csv(PROCESSED + 'fema_noaa_merged.csv', index=False)
print(f'Saved: fema_noaa_merged.csv ({len(merged):,} rows)')